In [ ]:
import os
from crewai import LLM, Agent, Process, Crew, Task
from dotenv import load_dotenv
from llama_index_client.types import job_names

load_dotenv()

In [ ]:
MOONSHOT_API_KEY = os.getenv("MOONSHOT_API_KEY")

In [ ]:
llm = LLM(
    model="openai/kimi-k2.6",
    api_key=MOONSHOT_API_KEY,
    base_url="https://api.moonshot.ai/v1",
    # temperature=0.1,
)

In [ ]:
interviewer = input("Enter the name of the interviewer: ")
company = input("Enter the name of the company: ")
job_position = input("Enter the job position: ")
job_description = input("Enter the job description: ")

In [ ]:
# Interviewer AI Agent

interviewer_agent = Agent(
    role = f"You are an {interviewer}, who is the CEO of the company and is hiring for a seasoned AI Engineer",
    goal = f"Conduct a rigorous interview for the {job_position} role. Ask questions that test technical depth, system design, engineering judgment, and first-principles reasoning. The job description is : {job_description}",
    backstory = f"{interviewer} is an AI Expert, and a serial entrepreneur. They are a systems and first principles thinker. They previously also held senior management positions at large MNCs. They are also an expert at Python and Rust programming. You are simulating an interview being conducted by them.",
    llm = llm
)

In [ ]:
# Define the interview prep task

interview_prep_task = Task(
    description = f"Interview the candidate for the job {job_position} with the job description: {job_description}",
    expected_output = f"Ask only one question to the candidate that is relevant for the job {job_position}",
    agent = interviewer_agent
)

In [ ]:
# Interview Coach Agent
coach_agent = Agent(
    role = "interview_coach",
    goal = f"Help the candidate prepare for {job_position} with description : '{job_description}' by grading the relevance of the candidate's answer and offering suggestions for improvement.",
    backstory = "An expert at interview prep for tech roles",
    llm = llm
)

In [ ]:
# Coaching Task
coaching_task = Task(
    description = f"Give candidate feedback on the relevance and quality of their answer, with suggestions for improvement for the role of {job_position} with description: {job_description}.",
    expected_output = "A structured and concise response highlighting what they got right, where they can improve and any other suggestions for improvement. The response should be in extremely concise bullet points (not exceeding 5 bullets).",
    agent = coach_agent,
    context = [interview_prep_task],
    human_input = True,
)

In [ ]:
# candidate_answer = input("ENTER YOUR ANSWER: ")

In [ ]:
# AI Crew

crew = Crew(
    agents = [interviewer_agent, coach_agent],
    tasks = [interview_prep_task, coaching_task],
    verbose = True,
    process = Process.sequential
)

In [ ]:
result = await crew.kickoff_async()